In [1]:
# Step 1 - Load environment variables and API key


import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if api_key:
    print("API key loaded successfully!")
    print(f"Key starts with: {api_key[:8]}...")
else:
    print("API key missing - check your .env file")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    groq_api_key=api_key
)


response = llm.invoke("Say hello and introduce yourself briefly")
print("\nLLM Response:")
print(response.content)

e:\Random-Projects\Legal Assist Chatbot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


API key loaded successfully!
Key starts with: gsk_c4wF...

LLM Response:
Hello. I'm an artificial intelligence model known as Llama. I'm here to provide information and help with your questions to the best of my abilities.


In [2]:
# Step 2 - Load PDF documents from data folder

from langchain_community.document_loaders import PyPDFDirectoryLoader

loader = PyPDFDirectoryLoader("../data/")
documents = loader.load()

print(f"Total pages loaded: {len(documents)}")
print(f"\nFirst document snippet:")
print(documents[0].page_content[:300])

Total pages loaded: 3101

First document snippet:
THE CONSTITUTION OF INDIA 
[As on       May, 2022] 
2022


In [3]:
# Step 3 - Split documents into 

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")
print(f"\nExample chunk:")
print(chunks[0].page_content)
print(f"\nChunk metadata:")
print(chunks[0].metadata)

Total chunks created: 11694

Example chunk:
THE CONSTITUTION OF INDIA 
[As on       May, 2022] 
2022

Chunk metadata:
{'producer': 'iLovePDF', 'creator': 'PyPDF', 'creationdate': '', 'moddate': '2022-08-08T10:18:39+00:00', 'source': '..\\data\\2023050195.pdf', 'total_pages': 404, 'page': 0, 'page_label': '1'}


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Connecting to existing ChromaDB...")
vectorstore = Chroma(
    persist_directory="../chroma_db",
    embedding_function=embeddings
)

print(f"Connected! Total vectors: {vectorstore._collection.count()}")

Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2921.85it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Connecting to existing ChromaDB...
Connected! Total vectors: 11694


In [10]:
# Step 5 - Create the retriever

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 8}
)

test_query = "What are the fundamental rights of a person?"
retrieved_docs = retriever.invoke(test_query)

print(f"Retrieved {len(retrieved_docs)} chunks\n")
for i, doc in enumerate(retrieved_docs):
    print(f"--- Chunk {i+1} ---")
    print(f"Source: {doc.metadata.get('source', 'unknown')}")
    print(f"Content: {doc.page_content[:200]}")
    print()

Retrieved 8 chunks

--- Chunk 1 ---
Source: data\Justice_K_S_Puttaswamy_Retd_And_Anr_vs_Union_Of_India_And_Ors_on_24_August_2017.PDF
Content: short, the fundamental rights, subject to social control, have been incorporated in the
rule of law…„180 (emphasis supplied) The learned Judge emphasised the position of
the fundamental rights thus:
ƒ

--- Chunk 2 ---
Source: data\Maneka_Gandhi_vs_Union_Of_India_on_25_January_1978.PDF
Content: and educational rights , (vi) right to property, and (vii) right to constitutional
remedies. They are the rights of the people preserved by our Constitution,
'Fundamental rights' are the modem name fo

--- Chunk 3 ---
Source: data\Justice_K_S_Puttaswamy_Retd_And_Anr_vs_Union_Of_India_And_Ors_on_24_August_2017.PDF
Content: declared as a fundamental freedom as follows:
ƒ10. Human dignity Everyone has inherent dignity and the right to have their dignity
respected and protected.
12. Freedom and security of the person (1) E

--- Chunk 4 ---
Source: data\Kesavana

In [12]:
# Step 6 - Test with a question

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Provide the prompt format
template = """
You are an expert legal assistant specializing in Indian law, 
Supreme Court judgments and constitutional law.

Use the following context from actual legal documents to answer 
the question. Be specific and cite article numbers, section 
numbers, or case names where available.

If the context contains partial information, use it to give 
the best possible answer. Only say you don't have enough 
information if the context has absolutely nothing relevant.

Context:
{context}

Question: {question}

Answer (be specific, mention article/section numbers if available):
"""

prompt = ChatPromptTemplate.from_template(template)

# 2. Add a helper function to merge chunks of text together
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 3. Create the modern LCEL RAG chain
qa_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 4. Test query
query = "What is the punishment for murder under Section 302 IPC?"
print("QUESTION:")
print(query)

# Notice we use .invoke() directly with the string now!
result = qa_chain.invoke(query)

print("\nANSWER:")
print(result)


QUESTION:
What is the punishment for murder under Section 302 IPC?

ANSWER:
The punishment for murder under Section 302 of the Indian Penal Code (IPC) is death, or imprisonment for life, and shall also be liable to fine. This is stated in the context as "Punishment—Death, or imprisonment for life, and fine—Cognizable—Non-bailable—Triable by Court of Session—Non-compoundable" under Section 302 IPC.
